# ChatTTS Pipeline Worker — Colab GPU

Runs the **ChatTTS worker** on Colab's free GPU (T4). Synthesises audio
directly using ChatTTS — no separate Flask server needed.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

**Set Colab Secrets** (🔑 icon, left sidebar):

| Secret | Value |
|--------|-------|
| `NGROK_TOKEN` | ngrok authtoken (dashboard.ngrok.com → Your Authtoken) |
| `SSH_HOST` | server IP / hostname |
| `SSH_USER` | SSH username |
| `SSH_KEY`  | full private key (paste `-----BEGIN ... KEY-----`) |
| `SSH_REMOTE_PATH` | remote output dir, e.g. `/opt/tts-node/outputs` |
| `SSH_PORT` | SSH port (default 22) |
| `GITHUB_TOKEN` | PAT for private repos (optional) |
| `GITHUB_USER`  | GitHub username (optional) |

**Speaker embedding** is generated from a seed on first run and saved to
Google Drive so the same voice is reused across sessions. Delete
`My Drive/tts-pipeline/chattts_speaker.pt` to generate a new voice.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Mount Google Drive (MP3 output + speaker embedding storage) ─────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR  = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR   = '/content/spool'
REF_DIR     = '/content/ref'
DRIVE_REF   = '/content/drive/MyDrive/tts-pipeline/chattts_speaker.pt'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
os.makedirs(REF_DIR,    exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── 2. Secrets ─────────────────────────────────────────────────────────────────
# Add these in the Secrets panel (🔑 icon, left sidebar):
#
#   GITHUB_TOKEN      → ghp_...  (Personal Access Token — private repos only)
#   GITHUB_USER       → your GitHub username
#   NGROK_TOKEN       → ngrok authtoken (dashboard.ngrok.com → Your Authtoken)
#
#   SSH_HOST          → server IP or hostname  e.g. 10.0.0.1
#   SSH_USER          → SSH username           e.g. root
#   SSH_KEY           → private key contents   (paste the full -----BEGIN ... KEY-----)
#   SSH_REMOTE_PATH   → remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          → SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'ngrok token:  {"ok" if NGROK_TOKEN else "MISSING"}')
print(f'SSH host:     {"ok" if SSH_HOST    else "MISSING"}')
print(f'SSH key:      {"ok" if SSH_KEY     else "MISSING"}')

In [ ]:
# ── 3. SSH setup + mount remote output dir via sshfs ──────────────────────────
import os, subprocess, textwrap, re

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)

def _reconstruct_pem(raw: str) -> str:
    key = raw.replace('\\n', '\n').strip()
    pattern = r'(-----BEGIN [^-]+-----)(.*?)(-----END [^-]+-----)'
    match = re.search(pattern, key, re.DOTALL)
    if match:
        header, body, footer = match.groups()
        body = ''.join(body.split())
        wrapped = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
        return f'{header}\n{wrapped}\n{footer}\n'
    return key + '\n'

with open(SSH_KEY_PATH, 'w') as f:
    f.write(_reconstruct_pem(SSH_KEY))
os.chmod(SSH_KEY_PATH, 0o600)

check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
if check.returncode != 0:
    print('Debug - First 100 chars of processed key:\n', open(SSH_KEY_PATH).read()[:100])
    raise RuntimeError(f'SSH Key Error: {check.stderr}')

with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
            ConnectTimeout 10
    """))

print(f'SSH key verified. Testing connection to {SSH_USER}@{SSH_HOST}...')
test_conn = subprocess.run(
    ['ssh', f'{SSH_USER}@{SSH_HOST}', f'mkdir -p {SSH_REMOTE_PATH}'],
    capture_output=True, text=True
)
if test_conn.returncode != 0:
    raise RuntimeError(f'Connection failed: {test_conn.stderr}')

!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_cmd = [
    'sshfs', f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}', SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3'
]
result = subprocess.run(mount_cmd, capture_output=True, text=True)
if result.returncode == 0:
    OUTPUT_DIR = SSHFS_MOUNT
    os.environ['OUTPUT_DIR'] = OUTPUT_DIR
    print(f'Mounted {SSH_REMOTE_PATH} → {SSHFS_MOUNT}')
else:
    print(f'sshfs mount failed: {result.stderr}')
    print('Falling back to Google Drive output.')

In [ ]:
# ── 4. Config — edit these values ─────────────────────────────────────────────
REDIS_URL = 'rediss://redis.example.com:6380'  # ← your Redis URL (rediss:// for TLS)
WORKER_ID = 'colab-chattts-1'

# ChatTTS synthesis settings
CHATTTS_SPEED        = 5      # 1–9, 5 = normal pace
CHATTTS_TEMPERATURE  = 0.3
CHATTTS_TOP_P        = 0.7
CHATTTS_TOP_K        = 20
CHATTTS_SPEAKER_SEED = 42     # change for a different voice; delete Drive speaker.pt too
CHATTTS_MAX_CHARS    = 300    # max chars per synthesis segment

os.environ['REDIS_URL']            = REDIS_URL
os.environ['WORKER_ID']            = WORKER_ID
os.environ['OUTPUT_DIR']           = OUTPUT_DIR
os.environ['SPOOL_DIR']            = SPOOL_DIR
os.environ['CHATTTS_REF_DIR']      = REF_DIR
os.environ['USE_GPU']              = 'true'
os.environ['CHATTTS_DEVICE']       = 'cuda'
os.environ['CHATTTS_SPEED']        = str(CHATTTS_SPEED)
os.environ['CHATTTS_TEMPERATURE']  = str(CHATTTS_TEMPERATURE)
os.environ['CHATTTS_TOP_P']        = str(CHATTTS_TOP_P)
os.environ['CHATTTS_TOP_K']        = str(CHATTTS_TOP_K)
os.environ['CHATTTS_SPEAKER_SEED'] = str(CHATTTS_SPEAKER_SEED)
os.environ['CHATTTS_MAX_CHARS']    = str(CHATTTS_MAX_CHARS)
os.environ['PROMETHEUS_PORT']      = '8004'
print('Config set.')

In [ ]:
# ── 5. Install system dependencies ────────────────────────────────────────────
!apt-get install -y ffmpeg build-essential > /dev/null 2>&1
print('ffmpeg + build-essential installed.')

In [ ]:
# ── 6. Install ChatTTS + GPU PyTorch ──────────────────────────────────────────
# Colab T4 runs CUDA 12.x — use the cu121 wheel for PyTorch.
# ChatTTS pulls in torch, torchaudio, transformers, vocos, etc. automatically.
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/2noise/ChatTTS.git
!pip install -q soundfile pydub redis prometheus_client
print('All packages installed.')

In [ ]:
# ── 7. Speaker embedding ───────────────────────────────────────────────────────
# The speaker embedding fixes the voice for every chunk in every book.
# It is saved to Google Drive so the same voice is reused across Colab sessions.
#
# To get a different voice: change CHATTTS_SPEAKER_SEED above, delete
# My Drive/tts-pipeline/chattts_speaker.pt, then re-run this cell.

import os, shutil, torch
import ChatTTS

LOCAL_SPK = os.path.join(REF_DIR, 'speaker.pt')

if os.path.exists(DRIVE_REF):
    shutil.copy(DRIVE_REF, LOCAL_SPK)
    print(f'Loaded speaker embedding from Drive ({os.path.getsize(LOCAL_SPK):,} bytes)')
else:
    print(f'Generating speaker embedding (seed={CHATTTS_SPEAKER_SEED}) ...')
    # Load model temporarily just to sample a speaker, then unload.
    # The main warm-up cell (9) will reload it on GPU properly.
    _tmp_chat = ChatTTS.Chat()
    _tmp_chat.load(source='huggingface', device=torch.device('cpu'), compile=False)
    torch.manual_seed(CHATTTS_SPEAKER_SEED)
    _spk = _tmp_chat.sample_random_speaker()
    torch.save(_spk, LOCAL_SPK)
    shutil.copy(LOCAL_SPK, DRIVE_REF)
    del _tmp_chat
    print(f'Speaker embedding saved to Drive and {LOCAL_SPK}')

print('Speaker embedding ready.')

In [ ]:
# ── 8. Fetch the ChatTTS worker script ────────────────────────────────────────
import subprocess, shutil

result = subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/wizardgang/audiobook-stack',
    '/content/abogen'
], capture_output=True, text=True)
print(result.stdout or result.stderr)

shutil.copy('/content/abogen/tts-node/chattts-worker/worker.py', '/content/worker.py')
print('worker.py ready.')

In [ ]:
# ── 9. Verify Redis + output directory ────────────────────────────────────────
import redis, os
from pathlib import Path

r = redis.from_url(os.environ['REDIS_URL'], decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

out = Path(os.environ['OUTPUT_DIR'])
out.mkdir(parents=True, exist_ok=True)
probe = out / '.write_test'
try:
    probe.touch()
    probe.unlink()
    print(f'Output dir writable:  {out}  ✓')
except OSError as e:
    print(f'Output dir NOT writable: {out}  ✗  ({e})')
    print('  → chunks will spool to', os.environ['SPOOL_DIR'])

spool = Path(os.environ['SPOOL_DIR'])
spooled = list(spool.rglob('*.mp3'))
if spooled:
    print(f'WARNING: {len(spooled)} MP3(s) stuck in spool from a previous run:')
    for f in spooled[:5]:
        print(f'  {f}')
else:
    print(f'Spool clean: {spool}')

In [ ]:
# ── 10. Warm up ChatTTS model (downloads weights ~1 GB on first run) ───────────
print('Loading ChatTTS model on GPU... (first run downloads ~1 GB, subsequent runs use cache)')

import sys, torch, numpy as np
import soundfile as sf
import io
from pydub import AudioSegment
sys.path.insert(0, '/content')

import ChatTTS
_chat = ChatTTS.Chat()
_chat.load(source='huggingface', device=torch.device('cuda'), compile=False)

# Load the persisted speaker embedding
_spk_emb = torch.load(os.path.join(REF_DIR, 'speaker.pt'), map_location='cpu', weights_only=True)

print('Running smoke-test synthesis...')
_params_infer = ChatTTS.Chat.InferCodeParams(
    spk_emb=_spk_emb,
    prompt='[speed_5]',
    temperature=CHATTTS_TEMPERATURE,
    top_P=CHATTTS_TOP_P,
    top_K=CHATTTS_TOP_K,
)
_wavs = _chat.infer(['Pipeline worker ready.'], params_infer_code=_params_infer)
_wav  = np.array(_wavs[0], dtype=np.float32)
_dur  = len(_wav) / 24000
print(f'Smoke test OK — generated {_dur:.2f}s of audio at 24000 Hz')
del _chat, _spk_emb, _wavs, _wav  # free memory; worker.py manages its own instance

In [ ]:
# ── 11. Start the ChatTTS worker ───────────────────────────────────────────────
# Runs until the session ends or you interrupt the cell.
# Pulls chunk jobs from pipeline:tts, writes MP3s to OUTPUT_DIR
# (or spools locally to SPOOL_DIR if the mount is offline).
%run /content/worker.py